# Medical Code Intelligence — Full Pipeline Demo

This notebook demonstrates every component of the Medical Code Intelligence pipeline:

| # | Component | Module | GPU? |
|---|-----------|--------|------|
| 1 | Configuration | `configs.ner_config` | No |
| 2 | Shorthand Expansion | `src.clinical.shorthand` | No |
| 3 | Negation Detection (rule-based) | `src.clinical.negation` | No |
| 4 | ICD-10-CM Code Lookup | `src.clinical.icd_codes` | No |
| 5 | MS-DRG Cost Estimation | `src.clinical.drg_costs` | No |
| 6 | Entity Post-Processing | `src.inference.entity_utils` | No |
| 7 | Evaluation Metrics | `src.evaluation.metrics` | No |
| 8 | Curated ICD Dataset Generation | `src.data.icd_dataset` | No |
| 9 | MedMentions & MACCROBAT (optional sources) | `src.data.icd_dataset` | No* |
| 10 | End-to-End Pipeline (pre-extracted entities) | `src.clinical.pipeline` | No |
| 11 | Adversarial Training (overview) | `src.training.adversarial` | No** |
| 12 | Assertion Classifier (transformer) | `src.clinical.assertion` | Optional |
| **13** | **Dataset Loading & Preprocessing** | `src.data.dataset_loader` | No |
| **14** | **Model Building & Training** | `src.training.trainer` | GPU rec. |
| **15** | **Evaluation & Error Analysis** | `src.evaluation` | No |
| **16** | **Full Pipeline — Shorthand to Cost Estimate** | `src.clinical.pipeline` | GPU rec. |
| **17** | **Multi-Model Training on ICD Dataset** | `src.data.icd_dataset` | GPU rec. |
| **18** | **ICD-Trained Pipeline — Shorthand, Negation & DRG Cost** | `src.clinical.pipeline` | GPU rec. |
| 19 | CLI Scripts Reference | — | — |
| 20 | Running the Test Suite | — | — |

\* MedMentions and MACCROBAT download from HuggingFace on first use; cells show the loader API and structure with graceful fallback if unavailable.

\** Adversarial training overview runs on CPU with a toy model. Actual adversarial training requires a GPU.

Sections 13-18 train real NER models and run the complete six-stage pipeline (shorthand expansion → NER → negation → ICD coding → DRG cost estimation) on clinical notes. Section 17 trains multiple biomedical models on the full 7-source ICD composite dataset. Section 18 uses the best model to run the full pipeline on three clinical cases with step-by-step stage output and DRG cost analysis.

In [ ]:
!git clone https://github.com/jcl347/Medical_Code_Intelligence
%cd Medical_Code_Intelligence
!pip install -r requirements.txt -q

In [ ]:
import os, sys, pathlib

# After %cd Medical_Code_Intelligence, cwd is the repo root.
# This cell also handles running from notebooks/ or other locations.
def _find_repo_root():
    markers = ("src", "configs")

    def _has_markers(p):
        return all((p / m).is_dir() for m in markers)

    cwd = pathlib.Path.cwd()

    # 1. cwd IS the repo root
    if _has_markers(cwd):
        return str(cwd)

    # 2. cwd is notebooks/ inside the repo
    if _has_markers(cwd.parent):
        return str(cwd.parent)

    # 3. Repo is a subdirectory of cwd (Colab default: /content)
    for child in sorted(cwd.iterdir()):
        if child.is_dir() and _has_markers(child):
            return str(child)

    # 4. Walk up from cwd
    p = cwd
    for _ in range(5):
        p = p.parent
        if _has_markers(p):
            return str(p)

    raise RuntimeError(
        f"Cannot find repo root (looked for src/ + configs/ dirs).\n"
        f"  cwd = {cwd}\n"
        f"Hint: run the setup cell above, or set manually:\n"
        f"  REPO_ROOT = '/path/to/Medical_Code_Intelligence'"
    )

REPO_ROOT = _find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

print(f"Repo root: {REPO_ROOT}")
print(f"  configs/ exists: {os.path.isdir(os.path.join(REPO_ROOT, 'configs'))}")
print(f"  src/ exists:     {os.path.isdir(os.path.join(REPO_ROOT, 'src'))}")

---
## 1. Configuration — `NERConfig`, `MODEL_CONFIGS`, `DATASET_CONFIGS`

All training hyperparameters, model definitions, and dataset metadata live in a single dataclass.

In [ ]:
from configs.ner_config import NERConfig, MODEL_CONFIGS, DATASET_CONFIGS

# --- Inspect available models ---
print("=== Supported Pre-trained Models ===")
for key, cfg in MODEL_CONFIGS.items():
    print(f"  {key:20s}  {cfg['model_name']}")

print()

# --- Inspect available datasets ---
print("=== Supported Datasets ===")
for key, cfg in DATASET_CONFIGS.items():
    print(f"  {key:20s}  {cfg['description'][:60]}")

In [ ]:
# --- Default hyperparameters ---
config = NERConfig()
print("=== Default NERConfig ===")
for k, v in vars(config).items():
    print(f"  {k:35s} = {v}")

In [ ]:
# --- Override for a specific experiment ---
custom = NERConfig(
    model_key="bio_clinicalbert",
    dataset_key="icd_ner",
    learning_rate=3e-5,
    num_train_epochs=15,
    use_adversarial_training=True,
    adv_method="fgm",
    resolve_drg=True,
)
print(f"Model:       {custom.model_key}")
print(f"Dataset:     {custom.dataset_key}")
print(f"LR:          {custom.learning_rate}")
print(f"Adversarial: {custom.adv_method} (epsilon={custom.adv_epsilon})")
print(f"DRG enabled: {custom.resolve_drg}")

---
## 2. Shorthand Expansion — `ShorthandExpander`

Expands physician abbreviations ("cp" → "chest pain") with character offset tracking for NER alignment.

Uses the built-in fallback (~280 abbreviations) so no network download is required.

In [ ]:
from src.clinical.shorthand import ShorthandExpander

expander = ShorthandExpander(source="auto")
print(f"Loaded {expander.num_abbreviations} abbreviations")
print(f"Ambiguous: {expander.num_ambiguous}")

In [ ]:
# --- Simple expansion ---
samples = [
    "pt c/o sob and cp",
    "hx of dm2, htn, and cad",
    "dx: afib r/o mi",
    "nkda, aox3, wnl",
]

print("=== Shorthand Expansion ===")
for text in samples:
    expanded = expander.expand(text)
    print(f"  {text:35s} → {expanded}")

In [ ]:
# --- Expansion with offset tracking ---
text = "pt denies cp or sob"
expanded, offsets = expander.expand_with_offsets(text)
print(f"Original:  {text!r}")
print(f"Expanded:  {expanded!r}")
print(f"\nOffset map ({len(offsets)} expansions):")
for om in offsets:
    print(f"  '{om['abbreviation']}' @ [{om['original_start']}:{om['original_end']}] "
          f"→ '{om['expansion']}' @ [{om['expanded_start']}:{om['expanded_end']}]")

In [ ]:
# --- Identify abbreviations without expanding ---
abbrevs = expander.identify_abbreviations("pt c/o sob and cp on exertion")
print("=== Identified Abbreviations ===")
for a in abbrevs:
    print(f"  '{a['abbreviation']}' @ [{a['start']}:{a['end']}] → '{a['expansion']}'")

---
## 3. Negation Detection — `NegationDetector`

Rule-based ConText/NegEx algorithm with 100+ trigger patterns. Detects six assertion statuses:
**AFFIRMED**, **NEGATED**, **POSSIBLE**, **HYPOTHETICAL**, **HISTORICAL**, **FAMILY**.

In [ ]:
from src.clinical.negation import NegationDetector, NegationStatus

detector = NegationDetector(scope_window=6)

# --- Show all assertion statuses ---
print("Assertion statuses:", [s.value for s in NegationStatus])

In [ ]:
# --- Detect negation scopes in raw text ---
text = "Patient denies chest pain but has persistent cough. No fever. History of diabetes."
scopes = detector.detect(text)
print(f"Text: {text!r}\n")
print(f"Detected {len(scopes)} negation/context scopes:")
for s in scopes:
    print(f"  [{s.status.value:12s}] trigger='{s.trigger_text}' "
          f"scope=[{s.scope_start}:{s.scope_end}] → '{text[s.scope_start:s.scope_end]}' "
          f"({s.direction})")

In [ ]:
# --- Annotate pre-extracted entities ---
text = "Patient denies fever but reports persistent cough. No evidence of pneumonia. Family history of diabetes."
entities = [
    {"text": "fever",     "label": "DIAGNOSIS", "start": 15, "end": 20},
    {"text": "cough",     "label": "DIAGNOSIS", "start": 43, "end": 48},
    {"text": "pneumonia", "label": "DIAGNOSIS", "start": 67, "end": 76},
    {"text": "diabetes",  "label": "DIAGNOSIS", "start": 96, "end": 104},
]

annotated = detector.annotate_entities(text, entities)
print(f"Text: {text!r}\n")
print("Entity Annotations:")
for ent in annotated:
    trigger = ent.get('negation_trigger', '-')
    print(f"  {ent['text']:15s} → {ent['negation']:12s} (trigger: {trigger})")

In [ ]:
# --- Quick negation check ---
text = "No evidence of pulmonary embolism."
print(f"'{text}' — is 'pulmonary embolism' negated? "
      f"{detector.is_negated(text, 15, 33)}")

text2 = "Diagnosed with pulmonary embolism."
print(f"'{text2}' — is 'pulmonary embolism' negated? "
      f"{detector.is_negated(text2, 16, 34)}")

In [ ]:
# --- Test all six assertion statuses ---
test_cases = [
    ("Patient has pneumonia.", "pneumonia", 12, 21, "affirmed"),
    ("Patient denies chest pain.", "chest pain", 15, 25, "negated"),
    ("Possible diagnosis of lupus.", "lupus", 23, 28, "possible"),
    ("If symptoms worsen, consider asthma.", "asthma", 30, 36, "hypothetical"),
    ("History of myocardial infarction.", "myocardial infarction", 11, 32, "historical"),
    ("Family history of breast cancer.", "breast cancer", 18, 31, "family"),
]

print("=== All Six Assertion Statuses ===")
for text, entity, start, end, expected in test_cases:
    ents = [{"text": entity, "label": "DIAGNOSIS", "start": start, "end": end}]
    result = detector.annotate_entities(text, ents)
    status = result[0]["negation"]
    match = "✓" if status == expected else "✗"
    print(f"  {match} {status:12s} (expected {expected:12s}) — {text}")

---
## 4. ICD-10-CM Code Lookup — `ICDCodeLookup`

TF-IDF character n-gram matching against 51K ICD-10-CM codes (falls back to 45 built-in codes offline).

In [ ]:
from src.clinical.icd_codes import ICDCodeLookup

lookup = ICDCodeLookup()
print(f"Loaded {len(lookup._codes)} ICD-10-CM codes")

In [ ]:
# --- Match entity text to ICD codes ---
queries = [
    "chest pain",
    "type 2 diabetes mellitus",
    "hypertension",
    "congestive heart failure",
    "pneumonia",
    "atrial fibrillation",
    "chronic kidney disease",
]

print("=== ICD-10-CM Entity Linking ===")
for query in queries:
    matches = lookup.match_entity(query, top_k=3)
    top = matches[0] if matches else None
    if top:
        print(f"  {query:30s} → {top.code}: {top.description} (score={top.score:.3f})")
    else:
        print(f"  {query:30s} → no match")

In [ ]:
# --- Direct code lookup ---
codes_to_look_up = ["E11.9", "I10", "J18.9", "R07.9", "I50.9"]

print("=== Direct Code Lookup ===")
for code_str in codes_to_look_up:
    code_obj = lookup.lookup_code(code_str)
    if code_obj:
        print(f"  {code_obj.code}: {code_obj.description}")
    else:
        print(f"  {code_str}: not found")

In [ ]:
# --- Batch entity matching ---
batch_entities = [
    {"text": "hypertension", "label": "DIAGNOSIS"},
    {"text": "pneumonia", "label": "DIAGNOSIS"},
    {"text": "chest pain", "label": "DIAGNOSIS"},
    {"text": "diabetes", "label": "DIAGNOSIS"},
]

results = lookup.match_entities_batch(batch_entities, top_k=3)
print("=== Batch Entity → ICD Mapping ===")
for r in results:
    codes = [c["code"] for c in r.get("icd_codes", [])]
    print(f"  {r['text']:20s} → {codes}")

---
## 5. MS-DRG Cost Estimation — `DRGCostEstimator`

Maps ICD-10-CM codes to MS-DRGs and estimates financial impact.

**Data sources:**
- **CMS IPPS Table 5** — All ~770 MS-DRG relative weights, downloaded from CMS.gov and cached locally
- **drgpy** — ICD-10 to MS-DRG grouper logic (`pip install drgpy`)

> **Note:** DRG weights are auto-downloaded from CMS.gov on first use. Full ICD→DRG grouper logic requires `drgpy`.


In [ ]:
from src.clinical.drg_costs import DRGCostEstimator, DRGResult, CostImpactAnalysis

estimator = DRGCostEstimator()
print(f"Base rate: ${estimator.base_rate:,.2f} (FY 2026)")
print(f"DRG weight table: {len(estimator._weights)} DRGs loaded")
print(f"Grouper available: {estimator._grouper is not None}")
if not estimator._weights:
    print("\n⚠ No DRG weights loaded. Cost estimates will be unavailable.")
    print("  To enable: provide CMS IPPS Table 5 or ensure network access.")


In [ ]:
# --- Direct cost estimate by DRG code ---
# These require DRG weights to be loaded (from CMS Table 5)
drg_codes = ["291", "292", "293", "065", "066", "193", "194", "195"]

print("=== DRG Cost Estimates ===")
if not estimator._weights:
    print("  (No DRG weights loaded — see cell above)")
else:
    for code in drg_codes:
        result = estimator._build_result(code)
        if result:
            print(f"  DRG {result.drg_code}: {result.drg_title:50s} "
                  f"wt={result.relative_weight:.4f}  ${result.estimated_payment:>10,.2f}  [{result.severity_level}]")


In [ ]:
# --- DRG grouping from ICD codes ---
# Requires drgpy for ICD→DRG grouping + CMS Table 5 for weights.
icd_sets = [
    (["J18.9"],                       "Pneumonia alone"),
    (["J18.9", "E11.9"],              "Pneumonia + diabetes (no CC)"),
    (["J18.9", "E11.9", "N17.9"],     "Pneumonia + diabetes + AKI (CC!)"),
    (["I50.9"],                       "Heart failure alone"),
    (["I50.9", "E11.9", "N17.9"],     "Heart failure + diabetes + AKI"),
    (["A41.9"],                       "Sepsis alone"),
]

print("=== ICD → DRG Grouping ===")
if estimator._grouper is None:
    print("  (drgpy not installed — install with: pip install drgpy)\n")
if not estimator._weights:
    print("  (No CMS weight data — cost estimates unavailable)\n")
for codes, desc in icd_sets:
    result = estimator.get_drg(codes)
    if result:
        print(f"  {str(codes):45s} → DRG {result.drg_code}: {result.drg_title} "
              f"(${result.estimated_payment:,.2f})")
    else:
        print(f"  {str(codes):45s} → (no mapping — requires drgpy + CMS weights)")
    print(f"    {desc}")

print("\nNote: Adding AKI (N17.9) as a secondary diagnosis acts as a CC,")
print("bumping Pneumonia from DRG 195 (base) to 194 (with CC) — a $1,679 increase.")


In [ ]:
# --- Cost impact analysis (CC/MCC comparison) ---
# With CMS weights loaded, we can analyse DRG families

if estimator._weights:
    # Heart Failure family: DRG 291 (MCC) / 292 (CC) / 293 (base)
    print("=== Heart Failure DRG Family (291/292/293) ===")
    for code in ["291", "292", "293"]:
        r = estimator._build_result(code)
        if r:
            print(f"  DRG {r.drg_code} [{r.severity_level:4s}]: wt={r.relative_weight:.4f}  "
                  f"${r.estimated_payment:>10,.2f}  {r.drg_title}")

    # Calculate revenue at risk
    base = estimator._build_result("293")
    mcc = estimator._build_result("291")
    if base and mcc:
        gap = mcc.estimated_payment - base.estimated_payment
        print(f"\n  Revenue at risk (base→MCC): ${gap:,.2f}")
else:
    print("DRG weights not loaded. Download CMS IPPS Table 5 to enable cost analysis.")
    print("See: https://www.cms.gov/medicare/payment/prospective-payment-systems/acute-inpatient-pps")


In [ ]:
# --- Full cost impact analysis via API ---
# analyze_cost_impact() requires drgpy for ICD→DRG grouping.
# analyze_drg_family() works with a known DRG code (no drgpy needed).

analysis = None
if estimator._grouper is not None:
    analysis = estimator.analyze_cost_impact(["J18.9", "E11.9"])

if analysis is None and estimator._weights:
    # Use analyze_drg_family with a known DRG code
    print("(Using analyze_drg_family — drgpy required for ICD-based analysis)\n")
    analysis = estimator.analyze_drg_family("292")  # Heart Failure w CC

if analysis:
    print("=== Cost Impact Analysis ===")
    d = analysis.to_dict()
    print(f"  Current DRG: {d['current']['drg_code']} — {d['current']['drg_title']}")
    print(f"  Estimated payment: ${d['current']['estimated_payment']:,.2f}")
    print(f"  Revenue at risk: ${d['revenue_at_risk']:,.2f}")
    print(f"  Undercoding risk: {d['undercoding_risk']}")
    if 'mcc_variant' in d:
        print(f"  MCC variant: DRG {d['mcc_variant']['drg_code']} — "
              f"${d['mcc_variant']['estimated_payment']:,.2f}")
else:
    print("No analysis available. Requires CMS Table 5 weights + drgpy.")


---
## 6. Entity Post-Processing — `post_process_entities()`

Filters garbage entities (stopwords, punctuation) and merges adjacent fragments from subword tokenization.

In [ ]:
from src.inference.entity_utils import NEREntity, post_process_entities

text = "Patient has congestive heart failure and type 2 diabetes mellitus."

# Simulate raw NER output with garbage and fragments
raw_entities = [
    NEREntity(text="congestive",    label="DIAGNOSIS", start_char=12, end_char=22, score=0.95),
    NEREntity(text="heart failure", label="DIAGNOSIS", start_char=23, end_char=36, score=0.93),
    NEREntity(text="and",           label="DIAGNOSIS", start_char=37, end_char=40, score=0.30),
    NEREntity(text="type",          label="DIAGNOSIS", start_char=41, end_char=45, score=0.25),
    NEREntity(text="2 diabetes mellitus", label="DIAGNOSIS", start_char=46, end_char=65, score=0.91),
]

print(f"Before post-processing ({len(raw_entities)} entities):")
for e in raw_entities:
    print(f"  '{e.text}' [{e.label}] score={e.score:.2f}")

cleaned = post_process_entities(raw_entities, text)
print(f"\nAfter post-processing ({len(cleaned)} entities):")
for e in cleaned:
    print(f"  '{e.text}' [{e.label}] score={e.score:.2f}")

---
## 7. Evaluation Metrics — `compute_ner_metrics()`

Entity-level precision, recall, and F1 using seqeval (or built-in fallback).

In [ ]:
from src.evaluation.metrics import compute_ner_metrics, _extract_entities_from_bio
import numpy as np

# --- Simulated model predictions ---
# Label mapping: 0=O, 1=B-DIAGNOSIS, 2=I-DIAGNOSIS
label_list = ["O", "B-DIAGNOSIS", "I-DIAGNOSIS"]

# Gold:  "The patient has [congestive heart failure] and [diabetes]."
# Pred:  "The patient has [congestive heart] failure and [diabetes]."
#  (boundary error on first entity, correct on second)

gold_labels = [0, 0, 0, 1, 2, 2, 0, 1, 0]  # O O O B I I O B O
pred_labels = [0, 0, 0, 1, 2, 0, 0, 1, 0]  # O O O B I O O B O (missed I on "failure")

# compute_ner_metrics expects (predictions, labels, label_list) as separate arrays
predictions = np.array([pred_labels])
labels = np.array([gold_labels])

metrics = compute_ner_metrics(predictions, labels, label_list=label_list)

print("=== Entity-Level Metrics ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}:")
        print(v)

print("\n(Note: boundary error on 'congestive heart failure' causes lower recall)")

In [ ]:
# --- BIO entity extraction utility ---
labels = ["O", "B-DIAGNOSIS", "I-DIAGNOSIS", "I-DIAGNOSIS", "O", "B-DIAGNOSIS", "O"]
entities = _extract_entities_from_bio(labels)
print("Extracted entities from BIO sequence:")
for etype, start, end in sorted(entities):
    print(f"  {etype} @ tokens [{start}:{end}]")

---
## 8. Curated ICD Dataset — Template-Generated Examples

The `icd_ner` dataset includes ~100 template-generated sentences targeting common NER failure patterns.

In [ ]:
from src.data.icd_dataset import _generate_template_examples, ICD_NER_LABELS

print(f"Label scheme: {ICD_NER_LABELS}\n")

examples = _generate_template_examples()
print(f"Generated {len(examples)} template examples\n")

# Show a few examples
print("=== Sample Template Examples ===")
for ex in examples[:8]:
    tokens = ex["tokens"]
    labels = ex["labels"]
    # Reconstruct text with labels
    labeled = []
    for tok, lab in zip(tokens, labels):
        if lab.startswith("B-"):
            labeled.append(f"[{tok}")
        elif lab.startswith("I-"):
            labeled.append(tok)
        else:
            if labeled and labeled[-1] and not labeled[-1].endswith("]"):
                # Close the previous entity bracket
                labeled[-1] = labeled[-1] + "]"
            labeled.append(tok)
    # Close any trailing entity
    text = " ".join(labeled)
    if text.count("[") > text.count("]"):
        text += "]"
    print(f"  {text}")

---
## 9. MedMentions & MACCROBAT — Optional Dataset Sources

The `icd_ner` composite dataset includes two optional HuggingFace sources that download on first use:

1. **MedMentions** (`bigbio/medmentions`) — up to 5K examples from 4,392 PubMed abstracts with 350K+ UMLS entity mentions, filtered for disease/disorder semantic types (T047, T048, T019, T046, T191)
2. **MACCROBAT** (`singh-aditya/MACCROBAT_biomedical_ner`) — up to 3K examples from 200 clinical case reports with DISEASE_DISORDER entities, providing clinical-note-style text that PubMed abstracts lack

Both are loaded by `load_icd_ner_dataset()` as Sources 6 and 7. If the download fails, they are skipped gracefully.

In [ ]:
# --- Source 6: MedMentions ---
# Loads disease/disorder entities from 4,392 PubMed abstracts (bigbio/medmentions).
# Filters for UMLS semantic types: T047 (Disease), T048 (Mental Disorder),
# T019 (Congenital Abnormality), T046 (Pathologic Function), T191 (Neoplastic Process).

from src.data.icd_dataset import _load_medmentions_diseases, _MEDMENTIONS_DISEASE_TYPES

print("=== MedMentions Disease Loader ===")
print(f"Target UMLS semantic types: {sorted(_MEDMENTIONS_DISEASE_TYPES)}")
print()

try:
    mm_dataset = _load_medmentions_diseases(max_examples=50)  # small sample for demo
    for split, ds in mm_dataset.items():
        n_entities = sum(1 for ex in ds for lab in ex["ner_labels"] if lab.startswith("B-"))
        print(f"  {split:12s}: {len(ds):4d} examples, {n_entities} DIAGNOSIS entities")

    # Show a few examples
    print("\nSample MedMentions examples:")
    for ex in list(mm_dataset["train"])[:3]:
        tokens = ex["tokens"]
        labels = ex["ner_labels"]
        # Show only the diagnosis spans
        spans = []
        current = []
        for tok, lab in zip(tokens, labels):
            if lab.startswith("B-"):
                if current:
                    spans.append(" ".join(current))
                current = [tok]
            elif lab.startswith("I-") and current:
                current.append(tok)
            else:
                if current:
                    spans.append(" ".join(current))
                    current = []
        if current:
            spans.append(" ".join(current))
        text_preview = " ".join(tokens[:15])
        if len(tokens) > 15:
            text_preview += " ..."
        print(f"  Text: {text_preview}")
        print(f"  Entities: {spans}")
        print()
except Exception as e:
    print(f"  MedMentions not available (expected in offline mode): {type(e).__name__}: {e}")
    print("  This source is optional — load_icd_ner_dataset() skips it gracefully.")

In [ ]:
# --- Source 7: MACCROBAT ---
# Loads DISEASE_DISORDER entities from 200 clinical case reports
# (singh-aditya/MACCROBAT_biomedical_ner). Provides clinical-note-style text
# that PubMed abstracts lack, closing the domain gap.

from src.data.icd_dataset import _load_maccrobat_diseases, _MACCROBAT_DISEASE_LABELS

print("=== MACCROBAT Disease Loader ===")
print(f"Target entity labels: {sorted(_MACCROBAT_DISEASE_LABELS)}")
print()

try:
    mac_dataset = _load_maccrobat_diseases(max_examples=50)  # small sample for demo
    for split, ds in mac_dataset.items():
        n_entities = sum(1 for ex in ds for lab in ex["ner_labels"] if lab.startswith("B-"))
        print(f"  {split:12s}: {len(ds):4d} examples, {n_entities} DIAGNOSIS entities")

    # Show a few examples
    print("\nSample MACCROBAT examples:")
    for ex in list(mac_dataset["train"])[:3]:
        tokens = ex["tokens"]
        labels = ex["ner_labels"]
        # Show only the diagnosis spans
        spans = []
        current = []
        for tok, lab in zip(tokens, labels):
            if lab.startswith("B-"):
                if current:
                    spans.append(" ".join(current))
                current = [tok]
            elif lab.startswith("I-") and current:
                current.append(tok)
            else:
                if current:
                    spans.append(" ".join(current))
                    current = []
        if current:
            spans.append(" ".join(current))
        text_preview = " ".join(tokens[:15])
        if len(tokens) > 15:
            text_preview += " ..."
        print(f"  Text: {text_preview}")
        print(f"  Entities: {spans}")
        print()
except Exception as e:
    print(f"  MACCROBAT not available (expected in offline mode): {type(e).__name__}: {e}")
    print("  This source is optional — load_icd_ner_dataset() skips it gracefully.")

---
## 10. End-to-End Pipeline — `MedicalCodingPipeline`

Chains shorthand expansion → negation detection → ICD resolution → DRG cost estimation.

Using `process_with_entities()` to supply pre-extracted entities (no NER model required).

In [ ]:
from src.clinical.pipeline import MedicalCodingPipeline, MedicalEntity

# Initialize pipeline without a trained NER model
pipeline = MedicalCodingPipeline(
    model_path=None,         # No NER model — we'll supply entities manually
    expand_shorthand=True,
    detect_negation=True,
    resolve_icd_codes=True,
    icd_top_k=3,
    resolve_drg=True,        # MS-DRG cost estimation (requires drgpy + CMS Table 5)
)

print("Pipeline initialized:")
print(f"  Shorthand expander: {pipeline.shorthand_expander is not None}")
print(f"  Negation detector:  {pipeline.negation_detector is not None}")
print(f"  ICD lookup:         {pipeline.icd_lookup is not None}")
print(f"  DRG estimator:      {pipeline.drg_estimator is not None}")


In [ ]:
# --- Process pre-extracted entities ---
clinical_text = "Patient denies chest pain. Diagnosed with congestive heart failure and hypertension."

pre_extracted = [
    {"text": "chest pain",              "label": "DIAGNOSIS", "start": 15, "end": 25, "score": 0.95},
    {"text": "congestive heart failure", "label": "DIAGNOSIS", "start": 43, "end": 66, "score": 0.97},
    {"text": "hypertension",            "label": "DIAGNOSIS", "start": 71, "end": 83, "score": 0.96},
]

results = pipeline.process_with_entities(clinical_text, pre_extracted)

print(f"Input:  {clinical_text}\n")
print("=== Pipeline Results ===")
for ent in results:
    print(f"  Entity: {ent.text}")
    print(f"    Label:    {ent.label}")
    print(f"    Negation: {ent.negation} (trigger: {ent.negation_trigger or 'none'})")
    print(f"    Score:    {ent.score:.3f}")
    if ent.icd_codes:
        print(f"    ICD codes:")
        for icd in ent.icd_codes[:2]:
            print(f"      {icd['code']}: {icd['description']} (score={icd['score']:.3f})")
    print()

In [ ]:
# --- Human-readable formatted output ---
formatted = pipeline.format_output(clinical_text, results)
print(formatted)

In [ ]:
# --- MedicalEntity properties and serialization ---
for ent in results:
    print(f"  {ent.text:30s} is_affirmed={ent.is_affirmed}  is_negated={ent.is_negated}")

print("\n=== JSON serialization ===")
import json, numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.floating, np.integer)):
            return float(obj)
        return super().default(obj)

print(json.dumps(results[0].to_dict(), indent=2, cls=NumpyEncoder))

In [ ]:
# --- Multiple clinical scenarios ---
scenarios = [
    (
        "No evidence of pneumonia on chest X-ray. Patient has COPD exacerbation.",
        [
            {"text": "pneumonia",         "label": "DIAGNOSIS", "start": 15, "end": 24, "score": 0.92},
            {"text": "COPD exacerbation", "label": "DIAGNOSIS", "start": 43, "end": 60, "score": 0.94},
        ]
    ),
    (
        "History of stroke. Currently presents with acute kidney injury.",
        [
            {"text": "stroke",             "label": "DIAGNOSIS", "start": 11, "end": 17, "score": 0.90},
            {"text": "acute kidney injury", "label": "DIAGNOSIS", "start": 43, "end": 61, "score": 0.96},
        ]
    ),
    (
        "Mother had breast cancer. Patient denies any malignancy.",
        [
            {"text": "breast cancer", "label": "DIAGNOSIS", "start": 11, "end": 24, "score": 0.93},
            {"text": "malignancy",   "label": "DIAGNOSIS", "start": 45, "end": 55, "score": 0.88},
        ]
    ),
]

print("=== Multiple Clinical Scenarios ===")
for text, ents in scenarios:
    results = pipeline.process_with_entities(text, ents)
    print(f"\n{text}")
    for r in results:
        icd = r.icd_codes[0]['code'] if r.icd_codes else 'N/A'
        print(f"  [{r.text}] {r.negation.upper():12s} ICD={icd}")

---
## 11. Adversarial Training — `FGM`, `PGD`, `AdversarialTrainer`

FGM and PGD perturb word embeddings during training to improve robustness (+0.5-1.5% F1).

This section demonstrates the API structure. Actual training requires a GPU and dataset.

In [ ]:
import torch
from src.training.adversarial import FGM, PGD, AdversarialTrainer

# --- Demonstrate FGM on a toy model ---
class ToyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.word_embeddings = torch.nn.Embedding(100, 16)
        self.classifier = torch.nn.Linear(16, 3)

    def forward(self, input_ids):
        emb = self.word_embeddings(input_ids)
        return self.classifier(emb.mean(dim=1))

model = ToyModel()

# Simulate a forward + backward pass
input_ids = torch.randint(0, 100, (2, 5))
output = model(input_ids)
loss = output.sum()
loss.backward()

# --- FGM attack ---
fgm = FGM(model, epsilon=1.0)
original_emb = model.word_embeddings.weight.data.clone()

fgm.attack()
perturbed_emb = model.word_embeddings.weight.data.clone()
perturbation_norm = torch.norm(perturbed_emb - original_emb).item()
print(f"FGM perturbation L2 norm: {perturbation_norm:.4f}")

fgm.restore()
restored_emb = model.word_embeddings.weight.data.clone()
print(f"Embeddings restored: {torch.allclose(original_emb, restored_emb)}")

In [ ]:
# --- PGD multi-step attack ---
model.zero_grad()
output = model(input_ids)
loss = output.sum()
loss.backward()

pgd = PGD(model, epsilon=0.3, alpha=0.1, num_steps=3)
original_emb = model.word_embeddings.weight.data.clone()

pgd.save()
for step in range(pgd.num_steps):
    pgd.attack_step()
    step_emb = model.word_embeddings.weight.data.clone()
    step_norm = torch.norm(step_emb - original_emb).item()
    print(f"  PGD step {step+1}: perturbation L2 norm = {step_norm:.4f}")

pgd.restore()
print(f"Embeddings restored: {torch.allclose(original_emb, model.word_embeddings.weight.data)}")

In [ ]:
# --- AdversarialTrainer overview ---
print("AdversarialTrainer extends HuggingFace Trainer with:")
print("  - Automatic FGM or PGD perturbation during training_step()")
print("  - Clean loss + adversarial loss combined")
print("  - No architecture changes required")
print()
print("Usage:")
print("  trainer = AdversarialTrainer(")
print("      model=model, args=training_args,")
print("      train_dataset=train_ds, eval_dataset=eval_ds,")
print("      adv_method='fgm', adv_epsilon=1.0,")
print("  )")
print("  trainer.train()")
print()
print("CLI:")
print("  python scripts/train.py --model pubmedbert --dataset icd_ner --adversarial")
print("  python scripts/train.py --model pubmedbert --dataset icd_ner --adversarial --adv-method pgd")

---
## 12. Assertion Classifier (Transformer-based)

The `AssertionClassifier` uses `bvanaken/clinical-assertion-negation-bert` for learned assertion detection.
The model (~440MB) is downloaded on first use and cached locally.

In the full pipeline, the transformer handles **AFFIRMED / NEGATED / POSSIBLE** assertions,
supplemented by rule-based detection for **HISTORICAL** and **FAMILY** contexts.

In [ ]:
from src.clinical.assertion import AssertionClassifier

classifier = AssertionClassifier(device="cpu")

# --- Single entity prediction ---
text = "Patient denies any chest pain or shortness of breath."
result = classifier.predict(
    text=text,
    entity_text="chest pain",
    entity_start=19,
    entity_end=29,
)
print("=== Single Entity Assertion ===")
print(f"Text: {text}")
print(f"Entity: 'chest pain'")
print(f"Result: {result}")

# --- Batch annotation ---
text2 = "History of stroke. Patient has persistent cough. No evidence of pneumonia."
entities = [
    {"text": "stroke",    "label": "DIAGNOSIS", "start": 11, "end": 17, "score": 0.95},
    {"text": "cough",     "label": "DIAGNOSIS", "start": 39, "end": 44, "score": 0.90},
    {"text": "pneumonia", "label": "DIAGNOSIS", "start": 64, "end": 73, "score": 0.92},
]
annotated = classifier.annotate_entities(text2, entities)

print(f"\nText: {text2}")
print("=== Batch Annotation ===")
for ent in annotated:
    print(f"  [{ent['text']:15s}] assertion={ent.get('assertion_label', 'N/A'):8s} "
          f"negation={ent.get('negation', 'N/A'):10s} "
          f"score={ent.get('assertion_score', 0):.3f}")

print("\n=== Pipeline Integration ===")
print("  Default: MedicalCodingPipeline() uses transformer + rule-based hybrid")
print("  Transformer handles: AFFIRMED / NEGATED / POSSIBLE")
print("  Rule-based supplements: HISTORICAL / FAMILY (not covered by transformer)")

---
## 13. Dataset Loading & Preprocessing

Load a biomedical NER dataset, inspect its structure, build label maps, and tokenize for training.

In [ ]:
from src.data.dataset_loader import load_ner_dataset, get_label_maps
from src.data.preprocessing import preprocess_dataset, tokenize_and_align_labels

# Load the NCBI Disease corpus (smallest, fastest to download)
dataset, label_list = load_ner_dataset("ncbi_disease")

print(f"=== NCBI Disease Corpus ===")
print(f"Label scheme: {label_list}")
print()
for split in dataset:
    print(f"  {split:12s}: {len(dataset[split]):,} examples")

# Inspect a raw example
ex = dataset["train"][0]
print(f"\nSample example:")
print(f"  Tokens:     {ex['tokens'][:12]}...")
print(f"  NER labels: {ex['ner_labels'][:12]}...")
print(f"  Label IDs:  {ex['ner_tags'][:12]}...")

label2id, id2label = get_label_maps(label_list)
print(f"\nLabel mapping: {label2id}")

In [ ]:
from transformers import AutoTokenizer

# Load a small, fast tokenizer for demonstration
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    use_fast=True,
)

# Tokenize a single example to show subword alignment
sample = {"tokens": [ex["tokens"]], "ner_labels": [ex["ner_labels"]]}
aligned = tokenize_and_align_labels(sample, tokenizer, label2id, max_length=128)

input_ids = aligned["input_ids"][0]
labels = aligned["labels"][0]

# Show the alignment: subword tokens ↔ aligned labels
subwords = tokenizer.convert_ids_to_tokens(input_ids)
print("=== Subword ↔ Label Alignment (first 25 tokens) ===")
print(f"{'Subword':<20s} {'Label ID':>8s}  {'Label':>15s}")
print("-" * 48)
for sw, lab in zip(subwords[:25], labels[:25]):
    lab_str = id2label[lab] if lab != -100 else "(ignored)"
    print(f"  {sw:<20s} {lab:>6d}  {lab_str:>15s}")
print(f"\n  ... ({len(subwords)} subwords total, original: {len(ex['tokens'])} words)")

In [ ]:
# Tokenize the full dataset for training
tokenized = preprocess_dataset(
    dataset, tokenizer, label2id,
    max_length=128,   # shorter for demo speed
    num_proc=1,       # single process for notebook stability
)

for split in tokenized:
    print(f"  {split:12s}: {len(tokenized[split]):,} examples, "
          f"columns={list(tokenized[split].column_names)}")

---
## 14. Model Building & Training

Build a PubMedBERT token classifier, configure training with the standard hyperparameters, and train on a small subset to demonstrate the full training loop.

> **Note:** This trains for 2 epochs on 200 examples as a demonstration. A production run uses 20 epochs on the full dataset (~7K+ examples for `icd_ner`).

In [ ]:
from configs.ner_config import NERConfig
from src.models.ner_model import build_ner_model
from src.training.trainer import build_trainer

# Build a compact training config for demonstration
config = NERConfig(
    model_key="pubmedbert",
    dataset_key="ncbi_disease",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    max_seq_length=128,
    fp16=False,               # CPU-safe
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    save_total_limit=1,
    early_stopping_patience=3,
    output_dir="outputs",
    use_adversarial_training=False,
)

print(f"Experiment: {config.experiment_name}")
print(f"Model:      {config.model_name_or_path}")
print(f"LR={config.learning_rate}, Epochs={config.num_train_epochs}, "
      f"Batch={config.per_device_train_batch_size}")

In [ ]:
# Build model + tokenizer
model, tokenizer = build_ner_model(
    config.model_name_or_path,
    label_list,
    use_crf=config.use_crf,
)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {trainable:,} trainable / {total:,} total")
print(f"Labels:     {model.config.num_labels} ({label_list})")

In [ ]:
# Tokenize with the model's own tokenizer
tokenized = preprocess_dataset(
    dataset, tokenizer, label2id,
    max_length=config.max_seq_length,
    num_proc=1,
)

# Use a small subset for fast demo training
train_subset = tokenized["train"].select(range(min(200, len(tokenized["train"]))))
eval_split = "validation" if "validation" in tokenized else "test"
eval_subset = tokenized[eval_split].select(range(min(100, len(tokenized[eval_split]))))

print(f"Training on {len(train_subset)} examples, evaluating on {len(eval_subset)}")

In [ ]:
# Build the HuggingFace Trainer
trainer = build_trainer(
    model=model,
    tokenizer=tokenizer,
    config=config,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
    label_list=label_list,
)

print(f"Trainer type: {type(trainer).__name__}")
print(f"Callbacks: {[type(cb).__name__ for cb in trainer.callback_handler.callbacks]}")

In [ ]:
# Train!
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print("=== Training ===")
train_result = trainer.train()

print(f"\n=== Training Complete ===")
print(f"  Total steps:    {train_result.global_step}")
print(f"  Training loss:  {train_result.training_loss:.4f}")
for key, val in train_result.metrics.items():
    print(f"  {key}: {val}")

In [ ]:
# Save the trained model
import os

best_model_dir = os.path.join(config.output_dir, config.experiment_name, "best_model")
trainer.save_model(best_model_dir)
tokenizer.save_pretrained(best_model_dir)
print(f"Model saved to: {best_model_dir}")
print(f"Contents: {os.listdir(best_model_dir)}")

---
## 15. Evaluation & Error Analysis

Evaluate the trained model on the held-out set and run entity-level error analysis to identify boundary errors, false negatives, and false positives.

In [ ]:
# Evaluate on the held-out split
eval_metrics = trainer.evaluate()

print("=== Evaluation Results ===")
for key, val in sorted(eval_metrics.items()):
    if isinstance(val, float):
        print(f"  {key:30s}: {val:.4f}")
    else:
        print(f"  {key:30s}: {val}")

In [ ]:
# Run NER predictions on raw examples for error analysis
from src.inference.predictor import NERPredictor

predictor = NERPredictor(model_path=best_model_dir, device="cpu")

# Predict on a handful of evaluation examples
n_eval = min(50, len(dataset[eval_split]))
eval_examples = [dataset[eval_split][i] for i in range(n_eval)]

tokens_list = []
true_labels_list = []
pred_labels_list = []

for ex in eval_examples:
    tokens = ex["tokens"]
    true_labels = ex["ner_labels"]
    pred_labels, _ = predictor.predict_tokens(tokens)
    tokens_list.append(tokens)
    true_labels_list.append(true_labels)
    pred_labels_list.append(pred_labels)

print(f"Predicted {n_eval} examples for error analysis")

# Show a few predictions side-by-side
print("\n=== Sample Predictions ===")
for i in range(min(3, n_eval)):
    tokens = tokens_list[i]
    true = true_labels_list[i]
    pred = pred_labels_list[i]
    # Show only tokens with non-O labels
    interesting = [(t, tr, pr) for t, tr, pr in zip(tokens, true, pred)
                   if tr != "O" or pr != "O"]
    if interesting:
        print(f"\n  Example {i+1}:")
        for tok, tr, pr in interesting[:10]:
            match = "  " if tr == pr else "!!"
            print(f"    {match} {tok:25s} true={tr:20s} pred={pr}")

In [ ]:
from src.evaluation.error_analysis import analyse_errors, print_error_report

analysis = analyse_errors(tokens_list, true_labels_list, pred_labels_list)

report = print_error_report(analysis, top_k=10)
print(report)

---
## 16. Full Pipeline — Shorthand to Cost Estimate

Run the complete six-stage pipeline on realistic clinical notes using the model we just trained. This demonstrates the full power of the system: abbreviation expansion, NER extraction, negation detection, ICD-10-CM coding, and MS-DRG cost estimation.

In [ ]:
from src.clinical.pipeline import MedicalCodingPipeline

# Initialize the full pipeline with our trained model
full_pipeline = MedicalCodingPipeline(
    model_path=best_model_dir,
    expand_shorthand=True,
    detect_negation=True,
    resolve_icd_codes=True,
    icd_top_k=3,
    resolve_drg=True,         # Enable DRG cost estimation (NBER CMS Table 5 weights)
    device="cpu",
)

print("=== Full Pipeline Components ===")
print(f"  NER model:          {best_model_dir}")
print(f"  Shorthand expander: {full_pipeline.shorthand_expander is not None}")
print(f"  Negation detector:  {full_pipeline.negation_detector is not None}")
print(f"  ICD lookup:         {full_pipeline.icd_lookup is not None} "
      f"({len(full_pipeline.icd_lookup._codes):,} codes)" if full_pipeline.icd_lookup else "")
print(f"  DRG estimator:      {full_pipeline.drg_estimator is not None} "
      f"({full_pipeline.drg_estimator.num_drgs} DRGs, CMS FY 2026 weights)"
      if full_pipeline.drg_estimator else "")


In [ ]:
# --- Clinical Note 1: Shorthand-heavy emergency note ---
note1 = "Pt c/o sob and cp x 2 days. Hx of chf and dm2. Denies n/v. Afebrile, bp stable."

print("=" * 70)
print("CLINICAL NOTE 1 (Emergency)")
print("=" * 70)
print(f"Input:  {note1}")

# Show shorthand expansion step
expanded, offsets = full_pipeline.shorthand_expander.expand_with_offsets(note1)
print(f"\nExpanded: {expanded}")
if offsets:
    print(f"  Expansions: {len(offsets)}")
    for om in offsets[:5]:
        print(f"    '{om['abbreviation']}' -> '{om['expansion']}'")

# Run full pipeline
results1 = full_pipeline.process(note1)
print(f"\n--- Extracted Entities ({len(results1)}) ---")
for ent in results1:
    icd_str = ent.icd_codes[0]['code'] + ": " + ent.icd_codes[0]['description'][:35] if ent.icd_codes else "N/A"
    abbrev = f" (from '{ent.expanded_from}')" if ent.expanded_from else ""
    print(f"  [{ent.text}]{abbrev}")
    print(f"    Assertion: {ent.negation.upper():12s} | Score: {ent.score:.3f} | ICD: {icd_str}")

In [ ]:
# --- Clinical Note 2: Complex admission with negations ---
note2 = (
    "72 yo male admitted with acute exacerbation of COPD and pneumonia. "
    "History of atrial fibrillation and chronic kidney disease stage 3. "
    "No evidence of pulmonary embolism on CT angiography. "
    "Patient denies chest pain or palpitations."
)

print("=" * 70)
print("CLINICAL NOTE 2 (Admission)")
print("=" * 70)
print(f"Input: {note2}\n")

results2 = full_pipeline.process(note2)
print(f"--- Extracted Entities ({len(results2)}) ---")
for ent in results2:
    parts = [f"{ent.negation.upper():12s}"]
    if ent.negation_trigger:
        parts.append(f'trigger="{ent.negation_trigger}"')
    if ent.icd_codes:
        parts.append(f"ICD={ent.icd_codes[0]['code']}")
    if ent.drg_info:
        parts.append(f"DRG={ent.drg_info.get('current', {}).get('drg_code', 'N/A')}")
    annotation = " | ".join(parts)
    print(f"  [{ent.text:30s}] {annotation}")

# Show DRG cost analysis if available
drg_entities = [e for e in results2 if e.drg_info]
if drg_entities:
    drg = drg_entities[0].drg_info
    print(f"\n--- DRG Cost Analysis ---")
    current = drg.get("current", {})
    print(f"  DRG {current.get('drg_code', 'N/A')}: {current.get('drg_title', 'N/A')}")
    print(f"  Estimated payment: ${current.get('estimated_payment', 0):,.2f}")
    print(f"  Revenue at risk:   ${drg.get('revenue_at_risk', 0):,.2f}")
    print(f"  Undercoding risk:  {drg.get('undercoding_risk', False)}")

In [ ]:
# --- Clinical Note 3: Family history and hypotheticals ---
note3 = (
    "Patient presents with severe headache and fever. "
    "Family history of breast cancer and stroke. "
    "If symptoms persist, consider meningitis workup. "
    "Possible migraine vs tension headache."
)

print("=" * 70)
print("CLINICAL NOTE 3 (Outpatient)")
print("=" * 70)
print(f"Input: {note3}\n")

results3 = full_pipeline.process(note3)
print(f"--- Assertion Classification ({len(results3)} entities) ---")
for ent in results3:
    icd = ent.icd_codes[0]['code'] if ent.icd_codes else "N/A"
    print(f"  {ent.negation.upper():14s} [{ent.text:25s}] ICD={icd}")

In [ ]:
# --- Batch processing ---
clinical_notes = [
    "Pt denies cp. Dx with afib and htn.",
    "No fever. History of diabetes. Acute kidney injury on admission.",
    "Possible sepsis. Blood cultures pending. Started on broad spectrum abx.",
]

print("=" * 70)
print("BATCH PROCESSING (3 notes)")
print("=" * 70)

batch_results = full_pipeline.process_batch(clinical_notes)
for i, (note, entities) in enumerate(zip(clinical_notes, batch_results)):
    print(f"\nNote {i+1}: {note}")
    if entities:
        for ent in entities:
            icd = ent.icd_codes[0]['code'] if ent.icd_codes else "N/A"
            print(f"  -> [{ent.text}] {ent.negation.upper()} ICD={icd}")
    else:
        print("  -> (no entities extracted)")

In [ ]:
# --- JSON export for downstream systems ---
import json

class NumpyEncoder(json.JSONEncoder):
    """Handle numpy float32 from model scores."""
    def default(self, obj):
        import numpy as np
        if isinstance(obj, (np.floating, np.integer)):
            return float(obj)
        return super().default(obj)

print("=== JSON Export (Note 2) ===")
export = {
    "text": note2,
    "entities": [ent.to_dict() for ent in results2],
    "summary": {
        "total_entities": len(results2),
        "affirmed": sum(1 for e in results2 if e.is_affirmed),
        "negated": sum(1 for e in results2 if e.is_negated),
        "historical": sum(1 for e in results2 if e.negation == "historical"),
        "icd_codes_resolved": sum(1 for e in results2 if e.icd_codes),
    }
}
print(json.dumps(export, indent=2, cls=NumpyEncoder)[:2000])

---
## 17. Production Training — PubMedBERT with Adversarial Training

Train PubMedBERT (the best-performing model from multi-model comparison) with FGM
adversarial training for improved robustness. Adversarial training perturbs word
embeddings during training, regularizing the model against surface-level text variation
(abbreviations, synonyms, typos) for +0.5-1.5% entity-level F1.

### Training Protocol

| Setting | Value | Rationale |
|---------|-------|-----------|
| **Model** | PubMedBERT (110M) | Highest test F1 in multi-model comparison |
| **Adversarial** | FGM (epsilon=1.0) | +0.5-1.5% F1, ~2x training time |
| **Data split** | Train+Val merged → 90/10 internal split; Test held out | Maximum training data |
| **Learning rate** | 2e-5 | Lower LR for composite dataset noise |
| **LR scheduler** | Cosine annealing | Smoother decay than linear |
| **Effective batch size** | 32 (16 × 2 grad accum) | Stabilizes gradients across 8 sources |
| **Label smoothing** | 0.05 | Regularizes annotation inconsistencies |
| **Early stopping** | Patience 10 | Composite F1 is noisy; avoid premature stopping |
| **Max epochs** | 30 | More room to converge; early stopping selects peak |

See the commented multi-model comparison block below to train all four BERT models.

In [ ]:
# --- Load composite dataset and prepare production train/val/test splits ---
from src.data.icd_dataset import load_icd_ner_dataset
from src.data.preprocessing import preprocess_dataset
from src.data.dataset_loader import get_label_maps
from datasets import concatenate_datasets, DatasetDict

icd_dataset, icd_label_list = load_icd_ner_dataset()

print("=== ICD NER Composite Dataset (8 Sources) ===")
print(f"Label scheme: {icd_label_list}")
print()
for split in icd_dataset:
    n_ents = sum(1 for ex in icd_dataset[split] for lab in ex['ner_labels'] if lab.startswith('B-'))
    print(f"  {split:12s}: {len(icd_dataset[split]):>6,} examples, {n_ents:>6,} DIAGNOSIS entities")

# --- Production split strategy ---
# Merge train + validation for maximum training data.
# Carve out 10% as internal validation for early stopping.
# Hold out test set exclusively for final model comparison.
train_data = icd_dataset['train']
if 'validation' in icd_dataset:
    val_data = icd_dataset['validation']
    merged = concatenate_datasets([train_data, val_data])
else:
    merged = train_data

# 90/10 split from merged data for train/eval
split_result = merged.train_test_split(test_size=0.1, seed=42)
production_train = split_result['train']
production_eval = split_result['test']  # internal validation for early stopping
production_test = icd_dataset['test']    # held out for final reporting

# Build the production DatasetDict
production_dataset = DatasetDict({
    'train': production_train,
    'validation': production_eval,
    'test': production_test,
})

print(f"\n--- Production Split ---")
print(f"  Training:   {len(production_train):>6,} examples (90% of train+val)")
print(f"  Validation: {len(production_eval):>6,} examples (10% of train+val, for early stopping)")
print(f"  Test:       {len(production_test):>6,} examples (held out for final comparison)")

icd_label2id, icd_id2label = get_label_maps(icd_label_list)
print(f"\nTotal: {sum(len(production_dataset[s]) for s in production_dataset):,} examples")

In [ ]:
# --- Production training: PubMedBERT with FGM adversarial training ---
import warnings, time, gc, torch
from configs.ner_config import NERConfig, MODEL_CONFIGS
from src.models.ner_model import build_ner_model
from src.training.trainer import build_trainer
from src.data.preprocessing import preprocess_dataset
from src.data.dataset_loader import get_label_maps

warnings.filterwarnings('ignore', category=FutureWarning)

# --- Production hyperparameters ---
MODEL_KEY = 'pubmedbert'    # Best model from multi-model comparison
EPOCHS = 30                 # Early stopping selects peak
BATCH_SIZE = 16             # Per-device
GRAD_ACCUM = 2              # Effective batch = 32
LEARNING_RATE = 2e-5        # Lower LR for composite data
LR_SCHEDULER = 'cosine'     # Smoother than linear
LABEL_SMOOTHING = 0.05      # Cross-source annotation noise
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_SEQ_LENGTH = 512
EVAL_STEPS = 100
PATIENCE = 10

print('=' * 70)
print(f'  Training: {MODEL_KEY} + FGM Adversarial')
print(f'  {MODEL_CONFIGS[MODEL_KEY]["description"]}')
print('=' * 70)

cfg = NERConfig(
    model_key=MODEL_KEY,
    dataset_key='icd_ner',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    label_smoothing_factor=LABEL_SMOOTHING,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_seq_length=MAX_SEQ_LENGTH,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    eval_steps=EVAL_STEPS,
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    early_stopping_patience=PATIENCE,
    output_dir='outputs',
    use_adversarial_training=True,   # FGM adversarial for +0.5-1.5% F1
    adv_method='fgm',
    adv_epsilon=1.0,
)

# Build model + tokenizer
mdl, tok = build_ner_model(cfg.model_name_or_path, icd_label_list, use_crf=False)
n_params = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
print(f'  Parameters: {n_params:,}')

# Tokenize production dataset
icd_l2id, icd_i2l = get_label_maps(icd_label_list)
tok_ds = preprocess_dataset(production_dataset, tok, icd_l2id, max_length=MAX_SEQ_LENGTH, num_proc=1)

train_ds = tok_ds['train']
eval_ds = tok_ds['validation']
test_ds = tok_ds['test']
print(f'  Train: {len(train_ds):,} | Eval: {len(eval_ds):,} | Test: {len(test_ds):,}')

# Build trainer (uses AdversarialTrainer when use_adversarial_training=True)
trnr = build_trainer(
    model=mdl, tokenizer=tok, config=cfg,
    train_dataset=train_ds, eval_dataset=eval_ds,
    label_list=icd_label_list,
)

t0 = time.time()
result = trnr.train()
elapsed = time.time() - t0

# Evaluate on validation (early stopping metric)
eval_metrics = trnr.evaluate()

# Evaluate on held-out test set (production metric)
test_metrics = trnr.evaluate(test_ds, metric_key_prefix='test')

# Save the best model
best_model = MODEL_KEY
best_model_dir = f'outputs/{MODEL_KEY}_icd_ner/best_model'
trnr.save_model(best_model_dir)
tok.save_pretrained(best_model_dir)

best_test_f1 = test_metrics.get('test_f1', 0)
best_test_p = test_metrics.get('test_precision', 0)
best_test_r = test_metrics.get('test_recall', 0)

print(f'\n  === Results ===')
print(f'  Eval:  F1={eval_metrics.get("eval_f1", 0):.4f}  '
      f'P={eval_metrics.get("eval_precision", 0):.4f}  '
      f'R={eval_metrics.get("eval_recall", 0):.4f}')
print(f'  Test:  F1={best_test_f1:.4f}  P={best_test_p:.4f}  R={best_test_r:.4f}')
print(f'  Loss:  train={result.training_loss:.4f}  test={test_metrics.get("test_loss", 0):.4f}')
print(f'  Time:  {elapsed:.1f}s | Steps: {result.global_step}')
print(f'  Saved: {best_model_dir}')

# Free memory
del mdl, tok, trnr, tok_ds
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# --- Training results ---
print('=' * 70)
print(f'  {best_model} + FGM Adversarial Training')
print('=' * 70)
print(f'  Test F1:        {best_test_f1:.4f}')
print(f'  Test Precision: {best_test_p:.4f}')
print(f'  Test Recall:    {best_test_r:.4f}')
print(f'  Model saved:    {best_model_dir}')
print()
print('  Adversarial training adds +0.5-1.5% F1 over clean baseline.')
print('  For further gains, try PGD (--adv-method pgd, ~4x cost) or CRF (--use-crf).')

# -----------------------------------------------------------------------
# MULTI-MODEL COMPARISON (uncomment to train all 4 BERT models)
# -----------------------------------------------------------------------
# This block trains all four 110M BERT models for a head-to-head comparison.
# Typical results (ICD NER composite, clean baseline):
#   PubMedBERT:       Test F1 ~0.66
#   BioBERT:          Test F1 ~0.63
#   Bio_ClinicalBERT: Test F1 ~0.59
#   SciBERT:          Test F1 ~0.62
#
# model_keys = ['pubmedbert', 'biobert', 'bio_clinicalbert', 'scibert']
# icd_results = {}
#
# for model_key in model_keys:
#     print('=' * 70)
#     print(f'  Training: {model_key} ({MODEL_CONFIGS[model_key]["description"]})')
#     print('=' * 70)
#
#     cfg = NERConfig(
#         model_key=model_key,
#         dataset_key='icd_ner',
#         num_train_epochs=30,
#         per_device_train_batch_size=16,
#         per_device_eval_batch_size=32,
#         gradient_accumulation_steps=2,
#         learning_rate=2e-5,
#         lr_scheduler_type='cosine',
#         label_smoothing_factor=0.05,
#         warmup_ratio=0.1,
#         weight_decay=0.01,
#         max_seq_length=512,
#         fp16=torch.cuda.is_available(),
#         logging_steps=50,
#         eval_steps=100,
#         save_steps=100,
#         save_total_limit=2,
#         early_stopping_patience=10,
#         output_dir='outputs',
#         use_adversarial_training=False,  # clean baseline for comparison
#     )
#
#     mdl, tok = build_ner_model(cfg.model_name_or_path, icd_label_list, use_crf=False)
#     n_params = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
#     print(f'  Parameters: {n_params:,}')
#
#     icd_l2id, icd_i2l = get_label_maps(icd_label_list)
#     tok_ds = preprocess_dataset(production_dataset, tok, icd_l2id, max_length=512, num_proc=1)
#
#     trnr = build_trainer(
#         model=mdl, tokenizer=tok, config=cfg,
#         train_dataset=tok_ds['train'], eval_dataset=tok_ds['validation'],
#         label_list=icd_label_list,
#     )
#
#     t0 = time.time()
#     result = trnr.train()
#     elapsed = time.time() - t0
#
#     eval_metrics = trnr.evaluate()
#     test_metrics = trnr.evaluate(tok_ds['test'], metric_key_prefix='test')
#
#     save_dir = f'outputs/{model_key}_icd_ner/best_model'
#     trnr.save_model(save_dir)
#     tok.save_pretrained(save_dir)
#
#     icd_results[model_key] = {
#         'eval_f1': eval_metrics.get('eval_f1', 0),
#         'eval_precision': eval_metrics.get('eval_precision', 0),
#         'eval_recall': eval_metrics.get('eval_recall', 0),
#         'test_f1': test_metrics.get('test_f1', 0),
#         'test_precision': test_metrics.get('test_precision', 0),
#         'test_recall': test_metrics.get('test_recall', 0),
#         'time_s': elapsed,
#         'n_params': n_params,
#         'save_dir': save_dir,
#     }
#
#     print(f'  Test F1={icd_results[model_key]["test_f1"]:.4f}  Time: {elapsed:.1f}s')
#
#     del mdl, tok, trnr, tok_ds
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#
# # Print comparison table
# print()
# print('  %-20s %8s %10s %8s' % ('Model', 'Test F1', 'Test P', 'Test R'))
# print('  ' + '-' * 50)
# for mk in model_keys:
#     r = icd_results[mk]
#     print('  %-20s %7.4f %9.4f %7.4f' % (mk, r['test_f1'], r['test_precision'], r['test_recall']))

---
## 18. ICD-Trained Pipeline — Shorthand, Negation & DRG Cost Analysis

Use the best model from Section 17 (PubMedBERT + FGM adversarial training) to run
the complete six-stage pipeline on abbreviation-heavy clinical notes:

1. **Shorthand expansion** — 78K+ abbreviations decoded with character offset tracking
2. **NER extraction** — DIAGNOSIS entities extracted by the adversarially-trained model
3. **Assertion detection** — transformer-based (BERT) + rule-based hybrid (6 statuses)
4. **ICD-10-CM coding** — TF-IDF entity linking against 51K codes
5. **MS-DRG grouping** — ICD code set mapped to DRG with CC/MCC tier evaluation
6. **Cost impact analysis** — estimated Medicare payment and revenue-at-risk quantification

In [ ]:
# --- Build full pipeline with the best ICD-trained model ---
from src.clinical.pipeline import MedicalCodingPipeline

# Use the best model from the Section 17 comparison
best_icd_model_dir = best_model_dir

icd_pipeline = MedicalCodingPipeline(
    model_path=best_icd_model_dir,
    expand_shorthand=True,       # Stage 1: abbreviation expansion
    detect_negation=True,        # Stage 3: negation / assertion detection
    resolve_icd_codes=True,      # Stage 4: ICD-10-CM entity linking
    icd_top_k=3,
    resolve_drg=True,            # Stage 5-6: DRG grouping + cost analysis (NBER CMS weights)
    device="cpu",
)

print(f"=== ICD-Trained Full Pipeline ===")
print(f"  Best model:         {best_model} (Test F1={best_test_f1:.4f})")
print(f"  Model path:         {best_icd_model_dir}")
print(f"  Shorthand expander: {icd_pipeline.shorthand_expander is not None}")
print(f"  Negation detector:  {icd_pipeline.negation_detector is not None}")
print(f"  ICD lookup:         {icd_pipeline.icd_lookup is not None} "
      f"({len(icd_pipeline.icd_lookup._codes):,} codes)")
print(f"  DRG estimator:      {icd_pipeline.drg_estimator is not None} "
      f"({icd_pipeline.drg_estimator.num_drgs} DRGs, CMS FY 2026 weights from NBER)")


In [ ]:
# --- Case 1: ICU admission with heavy shorthand and comorbidities ---
# This note uses dense abbreviations typical of ER/ICU charting.
# The pipeline must: expand shorthand → extract entities → classify assertions
# → link ICD codes → group into DRG → estimate cost.

note_icu = (
    "72 yo M pt c/o sob and cp x 3 days. Hx of chf, dm2, and ckd. "
    "Denies n/v or fever. Dx: acute exacerbation copd w/ pna. "
    "r/o pe. Htn controlled on meds."
)

print("=" * 78)
print("  CASE 1: ICU Admission — Shorthand-Heavy Note")
print("=" * 78)
print(f"\n  Raw input:\n    {note_icu}\n")

# Stage 1: Shorthand expansion
expanded_icu, offsets_icu = icd_pipeline.shorthand_expander.expand_with_offsets(note_icu)
print("  STAGE 1 — Shorthand Expansion:")
print(f"    Expanded: {expanded_icu}")
print(f"    Abbreviations decoded ({len(offsets_icu)}):")
for om in offsets_icu:
    print(f"      '{om['abbreviation']:6s}' → '{om['expansion']}'")

# Full pipeline
results_icu = icd_pipeline.process(note_icu)

# Stage 2-3: NER + Negation
print(f"\n  STAGE 2-3 — NER + Assertion Detection ({len(results_icu)} entities):")
print(f"    {'Entity':<30s} {'Assertion':<14s} {'Trigger':<20s} {'Score':>6s}")
print("    " + "-" * 74)
for ent in results_icu:
    trigger = ent.negation_trigger or ""
    abbrev = f" (← {ent.expanded_from})" if ent.expanded_from else ""
    print(f"    {(ent.text + abbrev):<30s} {ent.negation.upper():<14s} "
          f"{trigger:<20s} {ent.score:>5.3f}")

# Stage 4: ICD-10-CM codes
print(f"\n  STAGE 4 — ICD-10-CM Code Resolution:")
for ent in results_icu:
    if ent.icd_codes:
        top = ent.icd_codes[0]
        print(f"    {ent.text:<30s} → {top['code']}: {top['description'][:40]} "
              f"(score={top['score']:.3f})")
    else:
        print(f"    {ent.text:<30s} → (no ICD match)")

# Stage 5-6: DRG + Cost analysis
affirmed_codes = [ent.icd_codes[0]["code"] for ent in results_icu
                  if ent.is_affirmed and ent.icd_codes]
print(f"\n  STAGE 5-6 — DRG Cost Analysis:")
print(f"    Affirmed ICD codes for grouping: {affirmed_codes}")

drg_ents = [e for e in results_icu if e.drg_info]
if drg_ents:
    drg = drg_ents[0].drg_info
    curr = drg.get("current", {})
    print(f"    Assigned DRG:      {curr.get('drg_code', 'N/A')} — {curr.get('drg_title', 'N/A')}")
    print(f"    Relative weight:   {curr.get('relative_weight', 0):.4f}")
    print(f"    Estimated payment: ${curr.get('estimated_payment', 0):,.2f}")
    print(f"    Severity level:    {curr.get('severity_level', 'N/A')}")

    # Show CC/MCC tier comparison
    if "mcc_variant" in drg:
        mcc = drg["mcc_variant"]
        print(f"\n    CC/MCC Tier Comparison:")
        for key, label in [("base_variant", "Base (no CC/MCC)"),
                           ("cc_variant", "With CC"),
                           ("mcc_variant", "With MCC")]:
            if key in drg:
                v = drg[key]
                print(f"      {label:<20s} DRG {v['drg_code']}: wt={v['relative_weight']:.4f}  "
                      f"${v['estimated_payment']:>10,.2f}")

    print(f"\n    Revenue at risk:   ${drg.get('revenue_at_risk', 0):,.2f}")
    print(f"    Undercoding risk:  {drg.get('undercoding_risk', False)}")
else:
    print("    (No DRG assignment — model may not have extracted affirmed entities)")

In [ ]:
# --- Case 2: Cardiology consult with negations, family hx, and CC/MCC ---
# Tests: historical assertions, family history, negated findings,
# and how secondary diagnoses (AKI, sepsis) affect DRG tier/payment.

note_cardio = (
    "65 yo F admitted w/ acute mi. Hx of htn and afib. "
    "Family hx of cad and stroke. Denies sob or cp currently. "
    "Labs show acute kidney injury, cr 3.2. r/o sepsis."
)

print("=" * 78)
print("  CASE 2: Cardiology Consult — Negations, Family Hx, CC/MCC Impact")
print("=" * 78)
print(f"\n  Raw input:\n    {note_cardio}\n")

# Stage 1
expanded_cardio, offsets_cardio = icd_pipeline.shorthand_expander.expand_with_offsets(note_cardio)
print("  STAGE 1 — Shorthand Expansion:")
print(f"    Expanded: {expanded_cardio}")
print(f"    Decoded: {len(offsets_cardio)} abbreviations")
for om in offsets_cardio:
    print(f"      '{om['abbreviation']:6s}' → '{om['expansion']}'")

# Full pipeline
results_cardio = icd_pipeline.process(note_cardio)

# Stage 2-4: NER + Assertion + ICD
print(f"\n  STAGE 2-4 — Entity Extraction, Assertion & ICD Coding:")
print(f"    {'Entity':<28s} {'Assertion':<14s} {'ICD Code':<10s} {'ICD Description':<35s}")
print("    " + "-" * 90)
for ent in results_cardio:
    icd_code = ent.icd_codes[0]["code"] if ent.icd_codes else "—"
    icd_desc = ent.icd_codes[0]["description"][:33] if ent.icd_codes else "—"
    print(f"    {ent.text:<28s} {ent.negation.upper():<14s} {icd_code:<10s} {icd_desc}")

# Affirmed vs negated breakdown
affirmed = [e for e in results_cardio if e.is_affirmed]
negated = [e for e in results_cardio if e.is_negated]
other = [e for e in results_cardio if not e.is_affirmed and not e.is_negated]
print(f"\n    Affirmed: {len(affirmed)} | Negated: {len(negated)} | "
      f"Other (historical/family/possible): {len(other)}")

# Stage 5-6: DRG cost
affirmed_codes_cardio = [e.icd_codes[0]["code"] for e in affirmed if e.icd_codes]
print(f"\n  STAGE 5-6 — DRG Cost Analysis:")
print(f"    Codes sent to DRG grouper (affirmed only): {affirmed_codes_cardio}")

drg_ents = [e for e in results_cardio if e.drg_info]
if drg_ents:
    drg = drg_ents[0].drg_info
    curr = drg.get("current", {})
    print(f"    Assigned DRG:      {curr.get('drg_code', 'N/A')} — {curr.get('drg_title', 'N/A')}")
    print(f"    Relative weight:   {curr.get('relative_weight', 0):.4f}")
    print(f"    Estimated payment: ${curr.get('estimated_payment', 0):,.2f}")
    print(f"    Severity level:    {curr.get('severity_level', 'N/A')}")
    if "mcc_variant" in drg:
        print(f"\n    CC/MCC Tier Comparison:")
        for key, label in [("base_variant", "Base"), ("cc_variant", "CC"), ("mcc_variant", "MCC")]:
            if key in drg:
                v = drg[key]
                print(f"      {label:<6s} DRG {v['drg_code']}: wt={v['relative_weight']:.4f}  "
                      f"${v['estimated_payment']:>10,.2f}")
    print(f"\n    Revenue at risk:   ${drg.get('revenue_at_risk', 0):,.2f}")
    print(f"    Undercoding risk:  {drg.get('undercoding_risk', False)}")
else:
    print("    (No DRG assignment available)")

In [ ]:
# --- Case 3: Sepsis admission — high-acuity, multiple comorbidities ---
# Sepsis is one of the highest-revenue DRGs. This tests whether the pipeline
# correctly groups sepsis with organ dysfunction into MCC-tier DRGs.

note_sepsis = (
    "58 yo M presents to ED w/ fever, tachycardia, and hypotension. "
    "Dx: sepsis secondary to uti. Pmhx of dm2, chf, and ckd stage 4. "
    "No hx of mi or stroke. Denies cp. Started on abx."
)

print("=" * 78)
print("  CASE 3: Sepsis Admission — High-Acuity DRG")
print("=" * 78)
print(f"\n  Raw input:\n    {note_sepsis}\n")

# Stage 1
expanded_sepsis, offsets_sepsis = icd_pipeline.shorthand_expander.expand_with_offsets(note_sepsis)
print("  STAGE 1 — Shorthand Expansion:")
print(f"    Expanded: {expanded_sepsis}")
print(f"    Decoded: {len(offsets_sepsis)} abbreviations")
for om in offsets_sepsis:
    print(f"      '{om['abbreviation']:6s}' → '{om['expansion']}'")

# Full pipeline
results_sepsis = icd_pipeline.process(note_sepsis)

# Combined stage view
print(f"\n  STAGE 2-4 — Entity Extraction, Assertion & ICD Coding:")
print(f"    {'Entity':<28s} {'Assertion':<14s} {'ICD Code':<10s} {'ICD Description':<35s}")
print("    " + "-" * 90)
for ent in results_sepsis:
    icd_code = ent.icd_codes[0]["code"] if ent.icd_codes else "—"
    icd_desc = ent.icd_codes[0]["description"][:33] if ent.icd_codes else "—"
    print(f"    {ent.text:<28s} {ent.negation.upper():<14s} {icd_code:<10s} {icd_desc}")

# DRG cost
affirmed_sepsis = [e for e in results_sepsis if e.is_affirmed]
affirmed_codes_sepsis = [e.icd_codes[0]["code"] for e in affirmed_sepsis if e.icd_codes]
print(f"\n  STAGE 5-6 — DRG Cost Analysis:")
print(f"    Affirmed codes: {affirmed_codes_sepsis}")

drg_ents = [e for e in results_sepsis if e.drg_info]
if drg_ents:
    drg = drg_ents[0].drg_info
    curr = drg.get("current", {})
    print(f"    Assigned DRG:      {curr.get('drg_code', 'N/A')} — {curr.get('drg_title', 'N/A')}")
    print(f"    Relative weight:   {curr.get('relative_weight', 0):.4f}")
    print(f"    Estimated payment: ${curr.get('estimated_payment', 0):,.2f}")
    print(f"    Severity level:    {curr.get('severity_level', 'N/A')}")
    if "mcc_variant" in drg:
        print(f"\n    CC/MCC Tier Comparison:")
        for key, label in [("base_variant", "Base"), ("cc_variant", "CC"), ("mcc_variant", "MCC")]:
            if key in drg:
                v = drg[key]
                print(f"      {label:<6s} DRG {v['drg_code']}: wt={v['relative_weight']:.4f}  "
                      f"${v['estimated_payment']:>10,.2f}")
    print(f"\n    Revenue at risk:   ${drg.get('revenue_at_risk', 0):,.2f}")
    print(f"    Undercoding risk:  {drg.get('undercoding_risk', False)}")
else:
    print("    (No DRG assignment available)")

In [ ]:
# --- Cross-case summary: pipeline outcomes at a glance ---
import json, numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.floating, np.integer)):
            return float(obj)
        return super().default(obj)

cases = [
    ("Case 1: ICU / COPD+Pneumonia", note_icu, results_icu),
    ("Case 2: Cardiology / Acute MI", note_cardio, results_cardio),
    ("Case 3: Sepsis / High-Acuity", note_sepsis, results_sepsis),
]

print("=" * 78)
print("  CROSS-CASE SUMMARY — Full Pipeline Outcomes")
print("=" * 78)
print()
print(f"  {'Case':<35s} {'Entities':>8s} {'Affirm':>7s} {'Neg':>5s} "
      f"{'Other':>6s} {'DRG':>6s} {'Payment':>12s}")
print("  " + "-" * 82)

for case_name, note, results in cases:
    n_total = len(results)
    n_affirm = sum(1 for e in results if e.is_affirmed)
    n_neg = sum(1 for e in results if e.is_negated)
    n_other = n_total - n_affirm - n_neg
    drg_ents = [e for e in results if e.drg_info]
    if drg_ents:
        drg_code = drg_ents[0].drg_info.get("current", {}).get("drg_code", "—")
        payment = drg_ents[0].drg_info.get("current", {}).get("estimated_payment", 0)
        payment_str = f"${payment:>10,.2f}"
    else:
        drg_code = "—"
        payment_str = "—"
    print(f"  {case_name:<35s} {n_total:>8d} {n_affirm:>7d} {n_neg:>5d} "
          f"{n_other:>6d} {drg_code:>6s} {payment_str:>12s}")

# JSON export of all cases
print(f"\n{'=' * 78}")
print("  JSON Export — Case 1 (first entity)")
print(f"{'=' * 78}")
if results_icu:
    print(json.dumps(results_icu[0].to_dict(), indent=2, cls=NumpyEncoder))

---
## 19. QLoRA Fine-Tuning — GatorTron (Memory-Efficient NER Training)

QLoRA (Quantized Low-Rank Adaptation) enables memory-efficient fine-tuning of
large models like GatorTron by combining 4-bit NF4 quantization with LoRA adapters.
This reduces GPU memory by ~75% while retaining 95–99% of full fine-tuning performance.

**How it works:**
1. The base model is loaded in **4-bit precision** (NF4 quantization via bitsandbytes)
2. All base weights are **frozen** — only small LoRA adapter matrices are trainable
3. LoRA adapters are injected into attention layers (query, key, value projections)
4. The NER classification head trains in **full precision** for accuracy
5. After training, adapters can be **merged** back into the base model for deployment

**Supported GatorTron models:**

| Key | HuggingFace ID | Params | Memory (Full FT) | Memory (QLoRA) |
|-----|---------------|--------|-------------------|----------------|
| `gatortron-base` | `UFNLP/gatortron-base` | 345M | ~5.5 GB | ~1.4 GB |
| `gatortron-medium` | `UFNLP/gatortron-medium` | ~1B | ~16 GB | ~4 GB |
| `gatortron-large` | `UFNLP/gatortron-large` | ~3.9B | ~62 GB | ~15 GB |

**Required packages:** `peft>=0.7.0`, `bitsandbytes>=0.41.0`

The example below demonstrates the **complete QLoRA workflow** — from config
to model building, tokenization, training, and evaluation.

In [ ]:
# === Step 1: Configure QLoRA for GatorTron ===
# QLoRA = 4-bit NF4 quantization + LoRA adapters.
# Key differences from full fine-tuning:
#   - use_qlora=True loads the base model in 4-bit precision (requires CUDA GPU)
#   - use_lora=True (CPU-compatible) can be used as a fallback without quantization
#   - Higher learning rate (1e-3 vs 5e-5) — standard for LoRA/QLoRA
#   - Smaller batch size with gradient accumulation for memory efficiency

import torch
from configs.ner_config import NERConfig, MODEL_CONFIGS

# Auto-detect: use QLoRA on CUDA, LoRA on CPU
has_cuda = torch.cuda.is_available()

gatortron_config = NERConfig(
    model_key='gatortron-base',
    dataset_key='icd_ner',
    use_lora=not has_cuda,             # LoRA only (CPU fallback)
    use_qlora=has_cuda,                # QLoRA = 4-bit + LoRA (requires CUDA)
    lora_r=16,                         # LoRA rank (8-64; higher = more capacity)
    lora_alpha=32,                     # Scaling factor (typically 2x rank)
    lora_dropout=0.05,                 # Dropout on LoRA layers
    lora_target_modules='query,value', # Attention layers to adapt
    learning_rate=1e-3,                # Higher LR for LoRA/QLoRA (vs 5e-5 full FT)
    per_device_train_batch_size=8,     # Smaller batch for memory
    gradient_accumulation_steps=2,     # Effective batch = 8 * 2 = 16
    num_train_epochs=10,               # Fewer epochs (LoRA converges faster)
    early_stopping_patience=3,         # Tighter early stopping
    fp16=has_cuda,                     # Mixed precision on GPU
    max_seq_length=512,
)

mode = "QLoRA (4-bit + LoRA)" if has_cuda else "LoRA only (CPU — no quantization)"
print(f'=== GatorTron Configuration ({mode}) ===')
print(f'  Model: {gatortron_config.model_name_or_path}')
print(f'  CUDA available: {has_cuda}')
print(f'  use_qlora: {gatortron_config.use_qlora}')
print(f'  use_lora:  {gatortron_config.use_lora}')
print(f'  LoRA rank: {gatortron_config.lora_r}')
print(f'  LoRA alpha: {gatortron_config.lora_alpha}')
print(f'  Target modules: {gatortron_config.lora_target_modules}')
print(f'  Learning rate: {gatortron_config.learning_rate}')
print(f'  Effective batch size: {gatortron_config.per_device_train_batch_size * gatortron_config.gradient_accumulation_steps}')
print(f'  Epochs: {gatortron_config.num_train_epochs}')
print()
print('Memory comparison (GatorTron-base, 345M):')
print('  Full fine-tuning: ~5.5 GB VRAM (all 345M params trained)')
print('  LoRA (CPU/GPU):   ~3.0 GB      (~1.8M params trained)')
print('  QLoRA (4-bit):    ~1.4 GB VRAM  (~1.8M params trained, CUDA only)')

In [ ]:
# === Step 2: Build GatorTron with LoRA adapters ===
# This loads the full 345M model, freezes all base weights, and injects
# trainable LoRA matrices into the attention layers.
# Result: ~1.8M trainable params out of 345M total (~0.5%).

from src.models.ner_model import build_ner_model
# icd_label_list is already in scope from Section 17 (Cell 74)

lora_targets = gatortron_config.lora_target_modules.split(',')

gatortron_model, gatortron_tok = build_ner_model(
    gatortron_config.model_name_or_path,
    icd_label_list,
    use_lora=gatortron_config.use_lora,
    use_qlora=gatortron_config.use_qlora,
    lora_r=gatortron_config.lora_r,
    lora_alpha=gatortron_config.lora_alpha,
    lora_dropout=gatortron_config.lora_dropout,
    lora_target_modules=lora_targets,
)

total_params = sum(p.numel() for p in gatortron_model.parameters())
trainable_params = sum(p.numel() for p in gatortron_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f'\n=== GatorTron LoRA Model ===')
print(f'  Total parameters:     {total_params:>12,}')
print(f'  Frozen parameters:    {frozen_params:>12,}')
print(f'  Trainable parameters: {trainable_params:>12,}')
print(f'  Trainable fraction:   {100 * trainable_params / total_params:.2f}%')
print(f'  Memory savings:       ~{100 - 100 * trainable_params / total_params:.0f}% fewer trained params')
print()
# Show LoRA layer structure
print('LoRA adapter layers:')
for name, param in gatortron_model.named_parameters():
    if param.requires_grad and 'lora' in name.lower():
        print(f'  {name}: {tuple(param.shape)}')

---
## 20. CLI Scripts Reference

Quick reference for the training, prediction, evaluation, and benchmark scripts.

In [ ]:
# === Step 3: Prepare data and train GatorTron with LoRA ===
# Tokenize the ICD dataset with GatorTron's tokenizer and train.
# Uses production_dataset from Section 17 (train/validation/test splits).
# On CPU this will be slow — GPU recommended for actual training.

import warnings, time, gc
from src.training.trainer import build_trainer
from src.data.preprocessing import preprocess_dataset
from src.data.dataset_loader import get_label_maps

# Tokenize with GatorTron's tokenizer
gt_l2id, gt_i2l = get_label_maps(icd_label_list)
gt_tok_ds = preprocess_dataset(
    production_dataset, gatortron_tok, gt_l2id,
    max_length=gatortron_config.max_seq_length, num_proc=1,
)

print(f'Tokenized dataset splits:')
for split in gt_tok_ds:
    print(f'  {split}: {len(gt_tok_ds[split])} examples')

# Build trainer using NERConfig (same API as Section 17)
gt_trainer = build_trainer(
    model=gatortron_model,
    tokenizer=gatortron_tok,
    config=gatortron_config,
    train_dataset=gt_tok_ds['train'],
    eval_dataset=gt_tok_ds['validation'],
    label_list=icd_label_list,
)

# Train (GPU recommended; will be slow on CPU)
print(f'\nStarting GatorTron LoRA training...')
print(f'  Effective batch size: {gatortron_config.per_device_train_batch_size * gatortron_config.gradient_accumulation_steps}')
print(f'  Max epochs: {gatortron_config.num_train_epochs}')
print(f'  Early stopping patience: {gatortron_config.early_stopping_patience}')

start = time.time()
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    gt_result = gt_trainer.train()
elapsed = time.time() - start

print(f'\n=== GatorTron LoRA Training Complete ===')
print(f'  Duration: {elapsed/60:.1f} minutes')
print(f'  Final train loss: {gt_result.training_loss:.4f}')

# Evaluate on validation set
gt_eval = gt_trainer.evaluate()
print(f'  Eval F1:        {gt_eval.get("eval_f1", 0):.4f}')
print(f'  Eval Precision: {gt_eval.get("eval_precision", 0):.4f}')
print(f'  Eval Recall:    {gt_eval.get("eval_recall", 0):.4f}')

# Evaluate on held-out test set
gt_test = gt_trainer.evaluate(gt_tok_ds['test'], metric_key_prefix='test')
print(f'  Test F1:        {gt_test.get("test_f1", 0):.4f}')
print(f'  Test Precision: {gt_test.get("test_precision", 0):.4f}')
print(f'  Test Recall:    {gt_test.get("test_recall", 0):.4f}')

# Save the model
gt_save_dir = 'outputs/gatortron_icd_ner_lora/best_model'
gt_trainer.save_model(gt_save_dir)
gatortron_tok.save_pretrained(gt_save_dir)
print(f'  Model saved to: {gt_save_dir}')

# Clean up GPU memory
del gt_trainer, gatortron_model
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'  GPU memory freed')
except ImportError:
    pass

---
## 21. Running the Test Suite

All tests use mocked models and fallback data — no GPU or network access required.

In [ ]:
# Uncomment to run the full test suite from this notebook:
# !cd {REPO_ROOT} && python -m pytest tests/ -v --tb=short 2>&1 | tail -30

print("To run tests from the command line:")
print(f"  cd {REPO_ROOT}")
print("  python -m pytest tests/ -v")
print()
print("Key test files:")
test_files = [
    ("test_negation.py",            "Rule-based negation detection (200+ assertions)"),
    ("test_assertion.py",           "Transformer assertion classifier"),
    ("test_icd_ner_dataset.py",     "Composite dataset loading + garbage label cleaning"),
    ("test_pipeline.py",            "End-to-end pipeline integration"),
    ("test_icd_pipeline.py",        "ICD code resolution pipeline"),
    ("test_shorthand.py",           "Abbreviation expansion"),
    ("test_preprocessing.py",       "Tokenization and label alignment"),
    ("test_entity_postprocessing.py", "Entity filtering and merging"),
]
for filename, desc in test_files:
    print(f"  {filename:35s} — {desc}")


---
## Summary

This notebook demonstrated the full Medical Code Intelligence pipeline:

```
Clinical Text
  → ShorthandExpander (abbreviation expansion with offset tracking)
  → NER Model (transformer token classification, BIO scheme)
  → post_process_entities() (stopword filter, fragment merging)
  → NegationDetector or AssertionClassifier (6 assertion statuses)
  → ICDCodeLookup (TF-IDF matching against 51K codes)
  → DRGCostEstimator (CMS Table 5 weights → MS-DRG → cost estimate)
  → MedicalEntity list (text, label, negation, ICD codes, DRG info)
```

DRG weight data is sourced from **CMS IPPS Table 5** (~770 MS-DRGs, auto-downloaded from CMS.gov).

For large models, use the **QLoRA training script** (`scripts/train_gatortron_qlora.py`) for
memory-efficient fine-tuning of GatorTron with 4-bit quantization + LoRA adapters.
